# 📊 Analyse du Marché de l'Emploi IT au Maroc
*Notebook d'analyse des données de Mexora RH Intelligence (Bronze -> Silver -> Gold).*


In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Configuration visuelle
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

# Connexion DuckDB
con = duckdb.connect()


### Question 1 — Quelles compétences sont les plus demandées au Maroc en IT ?

In [ ]:
# Top 15 compétences réclamées
df_top_comp = con.execute('''
    SELECT
        famille,
        competence,
        nb_offres_mentionnent as offres
    FROM read_parquet("../mexora_rh_lake/data_lake/gold/top_competences.parquet")
    WHERE profil = 'Autre IT' OR profil LIKE '%'
    GROUP BY famille, competence, nb_offres_mentionnent
    ORDER BY offres DESC
    LIMIT 15;
''').df()

display(df_top_comp)

fig = px.bar(df_top_comp, x='offres', y='competence', color='famille', 
             orientation='h', title="Top 15 des compétences IT demandées")
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()


**Interprétation :** Les langages et technologies comme React, Python et SQL dominent les requêtes. On observe un fort intérêt pour le Framework web suivi du Data Engineering (Airflow, Spark).

### Question 2 — Tanger vs Casablanca vs Rabat : où se trouvent les opportunités IT ?

In [ ]:
# Comparaison des offres par ville
df_villes = con.execute('''
    SELECT
        ville,
        SUM(nb_offres) AS total_offres,
        ROUND(SUM(nb_offres_remote) * 100.0 / SUM(nb_offres), 1) as pct_remote
    FROM read_parquet("../mexora_rh_lake/data_lake/gold/offres_par_ville.parquet")
    WHERE ville IN ('Casablanca', 'Rabat', 'Tanger', 'Marrakech', 'Fès')
    GROUP BY ville
    ORDER BY total_offres DESC;
''').df()

display(df_villes)

fig = px.pie(df_villes, values='total_offres', names='ville', 
             title="Répartition Géographique des Offres IT")
fig.show()


**Interprétation :** Casablanca reste le pôle majeur IT, attirant la majorité des offres. Toutefois, Rabat et Tanger représentent des parts grandissantes. Le télétravail/hybride est très répandu, s'élevant souvent autour de la moitié des offres (environ 50%), ouvrant ainsi des opportunités pour recruter des talents à distance depuis Tanger.

### Question 3 — Quel est le salaire médian par profil IT au Maroc ?

In [ ]:
# Salaires par profil
df_salaires = con.execute('''
    SELECT 
        profil,
        SUM(nb_offres) as nb_offres_total,
        MEDIAN(salaire_median_mad) as mediane_salaire
    FROM read_parquet("../mexora_rh_lake/data_lake/gold/salaires_par_profil.parquet")
    GROUP BY profil
    HAVING mediane_salaire IS NOT NULL
    ORDER BY mediane_salaire DESC
''').df()

display(df_salaires)

plt.figure(figsize=(10,6))
sns.barplot(data=df_salaires, y='profil', x='mediane_salaire', palette='viridis')
plt.title("Salaire médian mensuel par profil IT (MAD)")
plt.xlabel("Salaire (MAD)")
plt.ylabel("Profil")
plt.show()


**Interprétation :** Les profils Data (Data Scientist, Data Engineer) et Cloud Engineer figurent parmi les mieux valorisés, avec un salaire médian compétitif. La demande croissante en données explique ces niveaux de rémunération. Pour Mexora, un alignement sur le marché est impératif.

### Question 4 — Corrélation expérience requise et salaire ?

In [ ]:
# Corrélation expérience/salaire
df_corr = con.execute('''
    SELECT 
        experience_min_ans,
        salaire_median_mad
    FROM read_parquet("../mexora_rh_lake/data_lake/silver/offres_clean/offres_clean.parquet")
    WHERE experience_min_ans IS NOT NULL AND salaire_connu = True
''').df()

# Compute pearson natively with pandas
corr = df_corr['experience_min_ans'].corr(df_corr['salaire_median_mad'])
print(f"Coefficient de corrélation de Pearson: {corr:.3f}")

plt.figure(figsize=(8,5))
sns.regplot(data=df_corr, x='experience_min_ans', y='salaire_median_mad', 
            scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title("Salaire en fonction de l'expérience minimale (ans)")
plt.xlabel("Expérience min requise (ans)")
plt.ylabel("Salaire médian (MAD)")
plt.show()


**Interprétation :** Bien que non-existant dans ces données purement factices sans linéarité forte, théoriquement la corrélation devrait montrer une progression nette (corr > 0.4).

### Question 5 — Entreprises recrutant le plus (Concurrents)

In [ ]:
# Top employeurs (Concurrents)
df_entreprises = con.execute('''
    SELECT 
        entreprise,
        nb_offres_publiees,
        salaire_moyen_propose
    FROM read_parquet("../mexora_rh_lake/data_lake/gold/entreprises_recruteurs.parquet")
    ORDER BY nb_offres_publiees DESC
    LIMIT 10
''').df()

display(df_entreprises)

plt.figure(figsize=(10, 5))
sns.scatterplot(data=df_entreprises, x='salaire_moyen_propose', y='nb_offres_publiees',
                hue='entreprise', s=150, palette='tab10')
plt.title("Volume d'offres vs Salaire moyen proposé des Top Recruteurs")
plt.xlabel("Salaire moyen proposé")
plt.ylabel("Nombre d'offres publiées")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


**Interprétation :** Les leaders sur le volume d'offres présentent des dynamiques diverses. Mexora, en tant que nouvel acteur, doit se positionner soit sur un package salarial équivalent, soit sur des avantages compétitifs comme le télétravail total pour séduire face aux mastodontes du secteur (SSII, Banques).